<a href="https://colab.research.google.com/github/gautam0309/university_faq_chatbot/blob/main/university_faq_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install nltk scikit-learn pandas

import pandas as pd
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from google.colab import files

In [ ]:
uploaded = files.upload()

data = pd.read_csv("faq_dataset_1000.csv")
data.head()

Saving faq_dataset_1000.csv to faq_dataset_1000 (2).csv


,Category,Question,Answer
0,Exams,Do you know when will semester exams start?,Semester exams will begin in December as per t...
1,Exams,When will semester exams start?,Semester exams will begin in December as per t...
2,Admissions,What is the last date to apply for admission?,The last date for admission is July 31st.
3,Exams,What is the passing criteria for each subject?,Students must score at least 40% marks to pass...
4,Exams,Can you tell me what is the passing criteria f...,Students must score at least 40% marks to pass...


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)
    tokens = [w for w in tokens if w.isalpha() and w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return ' '.join(tokens)

data['Processed_Question'] = data['Question'].apply(preprocess)
data.head()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,Category,Question,Answer,Processed_Question
0,Exams,Do you know when will semester exams start?,Semester exams will begin in December as per t...,know semester exam start
1,Exams,When will semester exams start?,Semester exams will begin in December as per t...,semester exam start
2,Admissions,What is the last date to apply for admission?,The last date for admission is July 31st.,last date apply admission
3,Exams,What is the passing criteria for each subject?,Students must score at least 40% marks to pass...,passing criterion subject
4,Exams,Can you tell me what is the passing criteria f...,Students must score at least 40% marks to pass...,tell passing criterion subject


In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(data['Processed_Question'])

In [ ]:
def chatbot_response(user_query):
    processed_query = preprocess(user_query)
    query_vec = vectorizer.transform([processed_query])
    similarity = cosine_similarity(query_vec, tfidf_matrix)

    idx = similarity.argmax()
    score = similarity[0][idx]

    if score < 0.3:
        return "I'm sorry, I don't have information about that. Please contact the admin office."

    return data.iloc[idx]['Answer']

In [ ]:
print("🎓 University FAQ Chatbot (type 'exit' to quit)\n")

while True:
    user_input = input("You: ")
    if user_input.lower() in ['exit', 'quit']:
        print("Chatbot: Goodbye! 👋")
        break
    print("Chatbot:", chatbot_response(user_input))

🎓 University FAQ Chatbot (type 'exit' to quit)

You: What is the passing criteria for each subject?
Chatbot: Students must score at least 40% marks to pass each subject.
You: When will semester exams start?
Chatbot: Semester exams will begin in December as per the academic calendar.
